# Setup

In [ ]:
import shutil, os
if os.path.exists('e_network_inequality'):
    shutil.rmtree('e_network_inequality')

!git clone -b main https://github.com/IgnacioOQ/e_network_inequality

In [ ]:
!pip install dill

In [ ]:
%cd e_network_inequality

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from utils.imports import *
from model.agents import BetaAgent, BayesAgent
from model.model import Model
from utils.network_utils import *
from networks.network_generation import *
from networks.variation_methods import *
from model.simulation_functions import *
from model.vectorized_simulation_functions import *
from functools import partial
import hashlib
import gc
from multiprocessing import get_context

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

dumping_path = '/content/drive/My Drive/Colab Projects/Data Driven ABMs/Data Sets/'
print("Current Directory:", dumping_path)

# Load Networks

In [ ]:
from model.vectorized_model import VectorizedModel
from model.vectorized_simulation_functions import run_vectorized_simulation_with_params
from functools import partial
from multiprocessing import Pool, cpu_count
import itertools
import pickle
import networkx as nx

num_cores = cpu_count()
print(f"Available CPU cores: {num_cores}")

with open('./networks/citation_data/pud_network.pkl', 'rb') as f:
    G_pud = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_pud.nodes())}
G_pud_indexed = nx.relabel_nodes(G_pud, mapping)
print(f"PUD network: {G_pud_indexed.number_of_nodes()} nodes, {G_pud_indexed.number_of_edges()} edges")

with open('./networks/citation_data/tobacco_network.pkl', 'rb') as f:
    G_tobacco = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_tobacco.nodes())}
G_tobacco_indexed = nx.relabel_nodes(G_tobacco, mapping)
print(f"Tobacco network: {G_tobacco_indexed.number_of_nodes()} nodes, {G_tobacco_indexed.number_of_edges()} edges")

with open('./networks/citation_data/ego_network.pkl', 'rb') as f:
    G_ego = pickle.load(f)
mapping = {node: idx for idx, node in enumerate(G_ego.nodes())}
G_ego_indexed = nx.relabel_nodes(G_ego, mapping)
print(f"Ego network: {G_ego_indexed.number_of_nodes()} nodes, {G_ego_indexed.number_of_edges()} edges")

# Study Example

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Study parameters
N_EXPERIMENTS = 100
UNCERTAINTY = 0.0001
N_RUNS = 100
MAX_STEPS = 100_000

def run_study(network, network_label, output_prefix):
    print(f"=== Study: {network_label} Network ===")
    
    param_dicts = [
        {"network": network,
         "n_experiments": N_EXPERIMENTS,
         "uncertainty": UNCERTAINTY,
         "seed": seed}
        for seed in range(N_RUNS)
    ]
    
    wrapper = partial(
        run_vectorized_simulation_with_params,
        tolerance=1e-5,
        tolerance_stopping=True,
        tstep_stopping=False,
        number_of_steps=MAX_STEPS,
        show_bar=False,
        agent_type="beta",
    )
    
    with Pool(num_cores) as pool:
        results = list(tqdm(
            pool.imap_unordered(wrapper, param_dicts),
            total=N_RUNS,
            desc=f"[{network_label}] Running simulations",
        ))
        
    # Save and return results
    df = pd.DataFrame(results)
    df.to_csv(dumping_path + f"{output_prefix}_results.csv", index=False)
    return df

## Disconnect from Runtime

In [ ]:
from datetime import datetime
import pytz
from IPython.display import Javascript

nyc_time = datetime.now(pytz.timezone('America/New_York'))
formatted_time = nyc_time.strftime('%Y-%m-%d %H:%M:%S %Z')
print(f"\u2705 Disconnected from runtime at: {formatted_time}")

display(Javascript('google.colab.kernel.disconnect()'))